# KlaDB Human Benchmark: Matched Architecture Evaluation

This notebook evaluates the PCBert-Kla baseline and MM-Kla token-gated architecture on the same locked, protein-group-disjoint human benchmark. Development uses training and validation only. Internal and study-held-out tests are evaluated once, after model settings and compute budget are fixed. Unreported lysines are putative negatives, not verified non-lactylated sites.

Required Kaggle inputs: a private dataset containing the paired `model_manifest.csv` and `split_audit.json` from `outputs/kladb_human_locked/`, and a Kaggle secret named `GITHUB_TOKEN` with read access to the private repository. Internet and two T4 GPUs are recommended. The notebook records the actual devices and commit used.

## 1. Runtime configuration

The capped pilot and full-epoch timing modes never evaluate either test partition. Final evaluation is disabled initially. Each final notebook version runs one seed from the predeclared set, with both architectures trained concurrently when two GPUs are available. This keeps each run below Kaggle's session limit; no multi-GPU training is implied.

In [ ]:
from pathlib import Path

RUN_CAPPED_PILOT = True
RUN_FULL_EPOCH_TIMING = False
RUN_FINAL_EVALUATION = False
FINAL_EPOCHS = 30
FINAL_PATIENCE = 5
FINAL_SEEDS = (42, 123, 777)
FINAL_SEED = 42  # Change only this value between final saved versions.
BATCH_SIZE = 4
SEED = 42
SAVE_FINAL_MODELS = True
LOCK_DIR = None  # Optional explicit Path within /kaggle/input.

WORKING_DIR = Path('/kaggle/working')
INPUT_DIR = Path('/kaggle/input')
REPO_DIR = WORKING_DIR / 'MM-Kla'
PROJECT_DIR = REPO_DIR / 'experiment' / 'pcbert_kla_clean'
OUTPUT_ROOT = WORKING_DIR / 'mm_kla_human_results'
REPO_URL = 'https://github.com/kaleabnega/MM-Kla.git'

if not WORKING_DIR.is_dir() or not INPUT_DIR.is_dir():
    raise RuntimeError('This notebook requires a Kaggle notebook runtime.')
if FINAL_EPOCHS < 1 or FINAL_PATIENCE < 0 or BATCH_SIZE < 1:
    raise ValueError('Training epochs, patience, and batch size are invalid.')
if not FINAL_SEEDS or len(FINAL_SEEDS) != len(set(FINAL_SEEDS)):
    raise ValueError('Final seeds must be a nonempty list without duplicates.')
if FINAL_SEED not in FINAL_SEEDS:
    raise ValueError('FINAL_SEED must be one of the predeclared FINAL_SEEDS.')
if RUN_FINAL_EVALUATION and (RUN_CAPPED_PILOT or RUN_FULL_EPOCH_TIMING):
    raise ValueError('Disable pilot and timing modes before final evaluation.')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Working directory:', WORKING_DIR)

## 2. Repository and environment

Private-clone credentials are supplied through Kaggle Secrets and are never placed in the clone URL or saved output. An existing checkout is reused unchanged. Package versions, GPU names, and the repository commit are recorded with the results.

In [ ]:
import importlib.metadata
import json
import os
import subprocess
import sys
import tempfile

if not REPO_DIR.exists():
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    if not token:
        raise RuntimeError('Kaggle secret GITHUB_TOKEN is missing or empty.')
    with tempfile.TemporaryDirectory() as temporary:
        askpass = Path(temporary) / 'askpass.sh'
        askpass.write_text(
            '#!/bin/sh\n'
            'case "$1" in\n'
            '  *Username*) printf "%s\\n" x-access-token ;;\n'
            '  *Password*) printf "%s\\n" "$MM_KLA_GITHUB_TOKEN" ;;\n'
            '  *) exit 1 ;;\n'
            'esac\n'
        )
        askpass.chmod(0o700)
        clone_env = os.environ.copy()
        clone_env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0',
                          'MM_KLA_GITHUB_TOKEN': token})
        try:
            clone = subprocess.run(
                ['git', '-c', 'credential.helper=', 'clone', '--depth', '1',
                 '--branch', 'main', REPO_URL, str(REPO_DIR)],
                env=clone_env, capture_output=True, text=True, check=False,
            )
        finally:
            clone_env.pop('MM_KLA_GITHUB_TOKEN', None)
            del token
    if clone.returncode:
        raise RuntimeError('Private clone failed; check Internet, secret access, and repository permissions. Git output is suppressed to protect the credential.')
elif not (REPO_DIR / '.git').is_dir():
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git checkout.')

if not (PROJECT_DIR / 'scripts' / 'run_replication.py').is_file():
    raise RuntimeError('The project runner is absent from the checkout.')
GIT_COMMIT = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
print('Repository commit:', GIT_COMMIT)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'],
               cwd=PROJECT_DIR, check=True)
import torch

GPU_COUNT = torch.cuda.device_count()
if GPU_COUNT < 1:
    raise RuntimeError('CUDA is unavailable. Enable Kaggle GPU acceleration before running models.')
runtime = {
    'git_commit': GIT_COMMIT, 'python': sys.version.split()[0],
    'cuda_device_count': GPU_COUNT,
    'cuda_devices': [
        {'name': torch.cuda.get_device_name(i),
         'memory_gb': round(torch.cuda.get_device_properties(i).total_memory / 2**30, 2)}
        for i in range(GPU_COUNT)
    ],
    'packages': {name: importlib.metadata.version(name) for name in
                 ('biopython', 'numpy', 'pandas', 'scikit-learn', 'torch',
                  'transformers', 'sentencepiece')},
}
(OUTPUT_ROOT / 'runtime.json').write_text(json.dumps(runtime, indent=2, sort_keys=True) + '\n')
print(json.dumps(runtime, indent=2, sort_keys=True))

## 3. Verify the locked benchmark

The pair of input files is checked against the locked v1 checksums before use. The runner then checks manifest integrity, study isolation, exact-window uniqueness, and protein-group separation. Source CSV and proteome downloads are not required for model training once these locked artifacts are available.

In [ ]:
import hashlib

EXPECTED_AUDIT_SHA256 = '2d1275577a430ff5758aa89c01a62c138c294ddc537bc67f671db3807856a0b8'
EXPECTED_MANIFEST_SHA256 = '842ff1b767d60f5645d3ba9c93150ec42ed9966dcc13672a679269327a25b95e'

if LOCK_DIR is None:
    candidates = [p.parent for p in INPUT_DIR.rglob('split_audit.json')
                  if (p.parent / 'model_manifest.csv').is_file()]
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one paired human lock dataset; found {len(candidates)}. Set LOCK_DIR explicitly if more than one is mounted.')
    LOCK_DIR = candidates[0]
else:
    LOCK_DIR = Path(LOCK_DIR)
FULL_LOCK = LOCK_DIR / 'split_audit.json'
FULL_MANIFEST = LOCK_DIR / 'model_manifest.csv'
if not FULL_LOCK.is_file() or not FULL_MANIFEST.is_file():
    raise FileNotFoundError('Locked input requires adjacent split_audit.json and model_manifest.csv.')
def file_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

if file_sha256(FULL_LOCK) != EXPECTED_AUDIT_SHA256:
    raise ValueError('The uploaded split audit differs from locked human benchmark v1.')
if file_sha256(FULL_MANIFEST) != EXPECTED_MANIFEST_SHA256:
    raise ValueError('The uploaded model manifest differs from locked human benchmark v1.')
full_audit = json.loads(FULL_LOCK.read_text())
if full_audit.get('train_positive_cap') != 0 or full_audit.get('holdout_pmid') != '41192556':
    raise ValueError('The input must be the uncapped study-held-out human lock.')
if RUN_FINAL_EVALUATION:
    prior_seeds = set()
    for prior_path in INPUT_DIR.rglob('final_seed_*.json'):
        prior = json.loads(prior_path.read_text())
        prior_seed = prior.get('seed')
        if prior_seed not in FINAL_SEEDS or prior_seed == FINAL_SEED or prior_seed in prior_seeds:
            raise ValueError(f'Unexpected or duplicate prior final seed in {prior_path}.')
        prior_seeds.add(prior_seed)
        if (prior.get('git_commit') != GIT_COMMIT
                or prior.get('planned_seeds') != list(FINAL_SEEDS)
                or prior.get('audit_sha256') != EXPECTED_AUDIT_SHA256
                or prior.get('manifest_sha256') != EXPECTED_MANIFEST_SHA256):
            raise ValueError(f'Prior final run uses a different code or dataset protocol: {prior_path}')
        previous = prior.get('settings', {})
        expected = {'epochs': FINAL_EPOCHS, 'patience': FINAL_PATIENCE,
                    'batch_size': BATCH_SIZE, 'optimizer': 'sgd',
                    'learning_rate': 0.003, 'weight_decay': 0.0,
                    'positive_weighting': 'none', 'monitor_metric': 'AUPRC',
                    'decision_threshold': 0.5}
        if any(previous.get(key) != value for key, value in expected.items()):
            raise ValueError(f'Prior final run uses different training settings: {prior_path}')
subprocess.run([sys.executable, 'scripts/run_replication.py', '--run', 'human-study-check',
                '--human-lock', str(FULL_LOCK)], cwd=PROJECT_DIR, check=True)
print('Verified locked benchmark:', LOCK_DIR)

## 4. Derive a training-only pilot

The pilot draws 500 positives and 1,500 negatives from the locked training partition with seed 42. It retains the exact full validation partition and never evaluates the two tests. It is only a pipeline and memory check; its score is not part of the paper comparison.

In [ ]:
import copy
import csv
import random

PILOT_DIR = WORKING_DIR / 'mm_kla_human_pilot_lock'
PILOT_LOCK = PILOT_DIR / 'split_audit.json'
if RUN_CAPPED_PILOT:
    with FULL_MANIFEST.open(newline='') as handle:
        reader = csv.DictReader(handle)
        fieldnames = reader.fieldnames
        rows = list(reader)
    train_pos = [r for r in rows if r['partition'] == 'train' and r['label'] == '1']
    train_neg = [r for r in rows if r['partition'] == 'train' and r['label'] == '0']
    if len(train_pos) < 500 or len(train_neg) < 1500:
        raise ValueError('Locked training partition is smaller than the pilot cohort.')
    rng = random.Random(SEED)
    pilot_rows = (rng.sample(train_pos, 500) + rng.sample(train_neg, 1500)
                  + [r for r in rows if r['partition'] != 'train'])
    pilot_rows.sort(key=lambda r: (r['partition'], r['accession'], int(r['position_1based'])))
    PILOT_DIR.mkdir(parents=True, exist_ok=True)
    pilot_manifest = PILOT_DIR / 'model_manifest.csv'
    with pilot_manifest.open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, lineterminator='\n')
        writer.writeheader()
        writer.writerows(pilot_rows)
    pilot_audit = copy.deepcopy(full_audit)
    pilot_audit['output_sha256'] = {'model_manifest.csv': file_sha256(pilot_manifest)}
    pilot_audit['pilot_only'] = True
    pilot_audit['parent_lock_sha256'] = file_sha256(FULL_LOCK)
    pilot_audit['train_positive_cap'] = 500
    pilot_audit['sampled_training_rows'] = 2000
    PILOT_LOCK.write_text(json.dumps(pilot_audit, indent=2, sort_keys=True) + '\n')
    subprocess.run([sys.executable, 'scripts/run_replication.py', '--run', 'human-study-check',
                    '--human-lock', str(PILOT_LOCK)], cwd=PROJECT_DIR, check=True)
    print('Pilot lock:', PILOT_LOCK)
else:
    print('Capped pilot disabled.')

## 5. Matched GPU runs

Both architectures use ProtBert, four encoder layers, SGD (learning rate 0.003), batch size 4, no positive weighting, validation AUPRC for checkpoint selection, and a fixed 0.5 decision threshold. Each architecture receives one GPU when two are available. Matched seeds are run as separate pairs. Process logs and wall times are saved under `/kaggle/working/mm_kla_human_results/`. A one-GPU runtime executes the models sequentially.

In [ ]:
import time

ARCHITECTURES = ('baseline', 'token_gated')
def run_pair(phase, lock_path, run_mode, epochs, patience, seed, save_models=False):
    phase_dir = OUTPUT_ROOT / phase / f'seed_{seed}'
    phase_dir.mkdir(parents=True, exist_ok=True)
    active = {}
    completed = {}
    log_handles = {}
    try:
        for index, architecture in enumerate(ARCHITECTURES):
            device = f'cuda:{index}' if GPU_COUNT >= 2 else 'cuda:0'
            command = [
                sys.executable, 'scripts/run_replication.py',
                '--run', run_mode, '--human-lock', str(lock_path),
                '--architecture', architecture, '--model-name', 'Rostlab/prot_bert',
                '--encoder-layers', '4', '--epochs', str(epochs),
                '--patience', str(patience), '--batch-size', str(BATCH_SIZE),
                '--device', device, '--optimizer', 'sgd',
                '--learning-rate', '0.003', '--weight-decay', '0',
                '--scheduler', 'none', '--positive-weighting', 'none',
                '--monitor-metric', 'AUPRC', '--decision-threshold', '0.5',
                '--seed', str(seed), '--output-dir', str(phase_dir),
            ]
            if run_mode == 'human-study-development':
                command.append('--no-save-predictions')
            if save_models:
                command.append('--save-models')
            log_path = phase_dir / f'{architecture}.log'
            log = log_path.open('w')
            log_handles[architecture] = log
            started = time.monotonic()
            process = subprocess.Popen(command, cwd=PROJECT_DIR, stdout=log,
                                       stderr=subprocess.STDOUT)
            active[architecture] = (process, started, device, log_path)
            print(f'Started {phase}: {architecture} on {device}', flush=True)
            if GPU_COUNT < 2:
                code = process.wait()
                if code:
                    raise RuntimeError(f'{architecture} failed (exit {code}).\n'
                                       + '\n'.join(log_path.read_text(errors='replace').splitlines()[-30:]))
                completed[architecture] = {'device': device,
                    'wall_seconds': round(time.monotonic() - started, 1),
                    'log': str(log_path)}
                del active[architecture]
        while active:
            for architecture, (process, started, device, log_path) in list(active.items()):
                code = process.poll()
                if code is None:
                    continue
                if code:
                    raise RuntimeError(f'{architecture} failed (exit {code}).\n'
                                       + '\n'.join(log_path.read_text(errors='replace').splitlines()[-30:]))
                completed[architecture] = {'device': device,
                    'wall_seconds': round(time.monotonic() - started, 1),
                    'log': str(log_path)}
                del active[architecture]
            if active:
                time.sleep(10)
    finally:
        for process, _, _, _ in active.values():
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
        for log in log_handles.values():
            log.close()
    record = {'phase': phase, 'seed': seed, 'run_mode': run_mode, 'epochs_requested': epochs,
              'lock_sha256': file_sha256(lock_path), 'runs': completed}
    (phase_dir / 'wall_times.json').write_text(json.dumps(record, indent=2) + '\n')
    print(json.dumps(record, indent=2))
    return record

pilot_times = (run_pair('pilot', PILOT_LOCK, 'human-study-development', 1, 0, SEED)
               if RUN_CAPPED_PILOT else None)

## 6. Uncapped one-epoch timing

This optional run uses every selected training example and the full validation partition, with no test inference. Its elapsed time is a more relevant scheduling estimate than the capped pilot because the model performs the complete training workload. First-epoch downloads and initialization are included in wall time.

In [ ]:
timing_times = (run_pair('full_epoch_timing', FULL_LOCK, 'human-study-development', 1, 0, SEED)
                if RUN_FULL_EPOCH_TIMING else None)
if timing_times:
    per_model_hours = {name: item['wall_seconds'] / 3600
                       for name, item in timing_times['runs'].items()}
    estimated_wall_hours = ((max(per_model_hours.values()) if GPU_COUNT >= 2
                             else sum(per_model_hours.values()))
                            * FINAL_EPOCHS * len(FINAL_SEEDS))
    print(f'Approximate wall time for {len(FINAL_SEEDS)} paired seeds at '
          f'{FINAL_EPOCHS} epochs: {estimated_wall_hours:.1f} h.')
    print('Early stopping and first-epoch overhead can change this estimate; '
          'Kaggle quota accounting may differ from wall-clock time.')
else:
    print('Uncapped timing run disabled.')

## 7. Final locked evaluation

This cell is disabled by default. For a final saved version, set `RUN_CAPPED_PILOT = False`, `RUN_FULL_EPOCH_TIMING = False`, and `RUN_FINAL_EVALUATION = True`; select exactly one `FINAL_SEED`. Run a separate saved version for each predeclared seed without changing the locked data or model settings. Each version trains both architectures, selects checkpoints by validation AUPRC, then evaluates the two test partitions without threshold tuning. A development-only full run is not required.

In [ ]:
final_times = None
if RUN_FINAL_EVALUATION:
    final_times = run_pair(
        'final', FULL_LOCK, 'human-study', FINAL_EPOCHS,
        FINAL_PATIENCE, FINAL_SEED, save_models=SAVE_FINAL_MODELS,
    )
if final_times is None:
    print('Final test evaluation remains disabled.')

## 8. Results from completed final runs

Metrics are read from model output files, not entered manually. A completed final version writes `mm_kla_human_results/final_seed_<seed>.json` alongside its raw logs, predictions, configs, and checkpoints. For the second and third versions, attach earlier saved notebook outputs as Kaggle inputs; the result files are discovered under `/kaggle/input`. Duplicate seeds or inconsistent datasets, code commits, and training settings are rejected. The combined summary is written only when all three predeclared seeds are present. Pilot and timing runs are excluded.

In [ ]:
import pandas as pd
from IPython.display import display

import math

METRIC_NAMES = ('ACC', 'AUC', 'AUPRC', 'F1', 'MCC', 'Pre', 'Rec', 'SP')
PROTOCOL_FIELDS = (
    'run', 'human_lock_sha256', 'model_name', 'sequence_format',
    'encoder_layers', 'freeze_encoder', 'epochs', 'patience', 'batch_size',
    'optimizer', 'learning_rate', 'weight_decay', 'sgd_momentum',
    'scheduler', 'positive_weighting', 'monitor_metric',
    'decision_threshold', 'calibrate_threshold', 'max_length',
    'fusion_dim', 'attention_dim', 'arch_dropout', 'ablation',
    'site_token_index',
)
EXPECTED_RUN_SETTINGS = {
    'run': 'human-study', 'human_lock_sha256': EXPECTED_AUDIT_SHA256,
    'model_name': 'Rostlab/prot_bert', 'encoder_layers': 4,
    'freeze_encoder': False, 'epochs': FINAL_EPOCHS,
    'patience': FINAL_PATIENCE, 'batch_size': BATCH_SIZE,
    'optimizer': 'sgd', 'learning_rate': 0.003,
    'weight_decay': 0.0, 'scheduler': 'none',
    'positive_weighting': 'none', 'monitor_metric': 'AUPRC',
    'decision_threshold': 0.5, 'calibrate_threshold': 'none',
    'ablation': 'none',
}
expected_partitions = full_audit['partitions_before_training_resampling']
EXPECTED_ROW_COUNTS = {
    'training_rows': full_audit['sampled_training_rows'],
    'validation_rows': expected_partitions['validation']['rows'],
    'test_rows': expected_partitions['internal_test']['rows'],
}

def build_seed_bundle(seed):
    run_root = OUTPUT_ROOT / 'final' / f'seed_{seed}' / 'human_study' / 'runs'
    records, artifacts, common_settings = [], {}, None
    for architecture in ARCHITECTURES:
        matches = []
        for config_path in run_root.glob('*/run_config.json'):
            config = json.loads(config_path.read_text())
            if config.get('architecture') == architecture and config.get('model_seed') == seed:
                matches.append((config_path.parent, config))
        if len(matches) != 1:
            raise RuntimeError(f'Expected one {architecture} final run for seed {seed}; found {len(matches)}.')
        run_dir, config = matches[0]
        for key, expected in {**EXPECTED_RUN_SETTINGS, **EXPECTED_ROW_COUNTS}.items():
            if config.get(key) != expected:
                raise ValueError(f'{architecture} seed {seed}: {key} differs from the locked protocol.')
        if config.get('extra_test_rows') != {'external_study': expected_partitions['external_test']['rows']}:
            raise ValueError(f'{architecture} seed {seed}: external test row count differs.')
        if not (1 <= config.get('selected_epoch', 0) <= config.get('completed_epochs', 0) <= FINAL_EPOCHS):
            raise ValueError(f'{architecture} seed {seed}: invalid completed or selected epoch.')
        if not config.get('save_predictions'):
            raise ValueError(f'{architecture} seed {seed}: test predictions were not saved.')
        settings = {name: config.get(name) for name in PROTOCOL_FIELDS}
        if common_settings is None:
            common_settings = settings
        elif settings != common_settings:
            raise ValueError(f'Seed {seed}: model training settings differ across architectures.')
        source_files = [run_dir / 'run_config.json',
                        run_dir / 'diagnostic_test_predictions.csv',
                        run_dir / 'external_study_predictions.csv']
        for split, filename in (('internal_test', 'test_metrics.json'),
                                ('study_held_out_test', 'external_study_metrics.json')):
            metrics_path = run_dir / filename
            metrics = json.loads(metrics_path.read_text())
            values = {name: float(metrics[name]) for name in METRIC_NAMES}
            if not all(math.isfinite(value) and -1 <= value <= 1 for value in values.values()):
                raise ValueError(f'{architecture} seed {seed}: invalid test metric.')
            records.append({'architecture': architecture, 'split': split,
                            'seed': seed, 'selected_epoch': config['selected_epoch'],
                            **values})
            source_files.append(metrics_path)
        for path in source_files:
            if not path.is_file():
                raise FileNotFoundError(f'Missing final-run artifact: {path}')
            artifacts[str(path.relative_to(OUTPUT_ROOT))] = file_sha256(path)
    return {
        'schema_version': 1, 'seed': seed, 'planned_seeds': list(FINAL_SEEDS),
        'git_commit': GIT_COMMIT, 'audit_sha256': EXPECTED_AUDIT_SHA256,
        'manifest_sha256': EXPECTED_MANIFEST_SHA256,
        'settings': common_settings, 'records': records, 'artifact_sha256': artifacts,
    }

if final_times is not None:
    bundle_path = OUTPUT_ROOT / f'final_seed_{FINAL_SEED}.json'
    bundle_path.write_text(json.dumps(build_seed_bundle(FINAL_SEED), indent=2, sort_keys=True) + '\n')
    print('Saved seed result bundle:', bundle_path)

bundle_paths = list(OUTPUT_ROOT.glob('final_seed_*.json'))
bundle_paths += list(INPUT_DIR.rglob('final_seed_*.json'))
bundles = {}
for path in bundle_paths:
    bundle = json.loads(path.read_text())
    if bundle.get('schema_version') != 1 or bundle.get('seed') not in FINAL_SEEDS:
        raise ValueError(f'Unexpected final result bundle: {path}')
    seed = bundle['seed']
    if seed in bundles:
        raise ValueError(f'Duplicate final result bundle for seed {seed}; detach the extra output version.')
    if bundle.get('planned_seeds') != list(FINAL_SEEDS):
        raise ValueError(f'Predeclared seed set differs in {path}.')
    if bundle.get('audit_sha256') != EXPECTED_AUDIT_SHA256 or bundle.get('manifest_sha256') != EXPECTED_MANIFEST_SHA256:
        raise ValueError(f'Locked dataset differs in {path}.')
    for relative, digest in bundle['artifact_sha256'].items():
        artifact = path.parent / relative
        if not artifact.is_file() or file_sha256(artifact) != digest:
            raise ValueError(f'Missing or modified final-run artifact: {artifact}')
    rows = bundle.get('records', [])
    expected_pairs = {(architecture, split) for architecture in ARCHITECTURES
                      for split in ('internal_test', 'study_held_out_test')}
    if len(rows) != 4 or {(row['architecture'], row['split']) for row in rows} != expected_pairs:
        raise ValueError(f'Incomplete architecture/split results in {path}.')
    if any(row['seed'] != seed for row in rows):
        raise ValueError(f'Inconsistent seed in {path}.')
    bundles[seed] = bundle

if not bundles:
    print('No completed final seed runs are available.')
else:
    protocols = {(bundle['git_commit'], json.dumps(bundle['settings'], sort_keys=True))
                 for bundle in bundles.values()}
    if len(protocols) != 1:
        raise ValueError('Code commits or training settings differ between final seeds.')
    results = pd.DataFrame(
        (row for seed in sorted(bundles) for row in bundles[seed]['records']),
        columns=['architecture', 'split', 'seed', 'selected_epoch', *METRIC_NAMES],
    )
    display(results.round(4))
    missing = sorted(set(FINAL_SEEDS) - set(bundles))
    if missing:
        print('Final summary pending. Missing predeclared seeds:', missing)
    else:
        summary = OUTPUT_ROOT / 'human_final_metrics.csv'
        results.to_csv(summary, index=False)
        grouped = results.groupby(['architecture', 'split'])[list(METRIC_NAMES)].agg(['mean', 'std'])
        display(grouped.round(4))
        grouped.to_csv(OUTPUT_ROOT / 'human_final_summary.csv')
        paired = results.pivot(index=['split', 'seed'], columns='architecture', values=list(METRIC_NAMES))
        differences = pd.DataFrame({name: paired[(name, 'token_gated')] - paired[(name, 'baseline')]
                                    for name in METRIC_NAMES})
        display(differences.round(4))
        differences.to_csv(OUTPUT_ROOT / 'human_paired_differences.csv')
        print('Combined metrics table:', summary)
        print('Raw run artifacts are retained in the saved Kaggle output versions.')